# Forensic AI - Colab Full-Quality Variant (LLaVA-1.5-7B 4-bit)

Run on Colab with **T4 GPU** (Runtime > Change runtime type > T4 GPU).
This notebook mirrors the local CPU pipeline but swaps Moondream2 for LLaVA-1.5-7B in 4-bit, which produces noticeably richer forensic descriptions.

Stages
1. Install deps (CUDA wheels)
2. Upload a traffic clip
3. YOLOv8 anomaly scan
4. LLaVA forensic analysis on the trigger window
5. Render incident report
6. (Optional) launch a Gradio UI for chat


In [ ]:
# 1. Verify GPU
!nvidia-smi

In [ ]:
# 2. Install deps - CUDA wheels for torch + 4-bit support
!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.43.3
!pip install -q ultralytics==8.2.103 opencv-python-headless==4.10.0.84 Pillow==10.4.0
!pip install -q scipy==1.13.1 gradio==4.44.0 einops==0.8.0

In [ ]:
# 3. Upload a traffic video. (Or skip and let the next cell synthesize one.)
import os, pathlib
from google.colab import files
uploaded = files.upload()  # browse to your local .mp4
video_path = list(uploaded.keys())[0] if uploaded else None
print('video_path =', video_path)

In [ ]:
# 3b. (Fallback) Synthesize a demo clip if nothing was uploaded.
if not video_path:
    import cv2, numpy as np
    out = '/content/demo_traffic.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    w, h, fps, dur = 640, 360, 30, 15
    writer = cv2.VideoWriter(out, fourcc, fps, (w, h))
    collision_frame = int(8.0 * fps)
    for i in range(fps * dur):
        f = np.full((h, w, 3), 50, dtype=np.uint8)
        x_a = int(60 + (min(i, collision_frame) / collision_frame) * 240)
        if i >= collision_frame:
            x_a = int(300 + (i - collision_frame) * 0.4)
        cv2.rectangle(f, (x_a, 200), (x_a + 80, 260), (0, 0, 220), -1)
        x_b = int(560 - (i / (fps * dur)) * 280)
        cv2.rectangle(f, (x_b, 215), (x_b + 30, 245), (220, 100, 30), -1)
        writer.write(f)
    writer.release()
    video_path = out
    print('synthesized', out)

In [ ]:
# 4. YOLO scan helper - reuses logic from the local repo.
from ultralytics import YOLO
import cv2, numpy as np

yolo = YOLO('yolov8n.pt')

def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1, ix2, iy2 = max(ax1,bx1), max(ay1,by1), min(ax2,bx2), min(ay2,by2)
    if ix2<=ix1 or iy2<=iy1: return 0.0
    inter = (ix2-ix1)*(iy2-iy1)
    return inter / (max(0,(ax2-ax1)*(ay2-ay1))+max(0,(bx2-bx1)*(by2-by1))-inter)

def scan(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    stride = max(1, int(fps / 5))
    idx, streak, trigger = 0, 0, None
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % stride == 0:
            r = yolo.predict(frame, verbose=False, conf=0.35)[0]
            boxes = []
            if r.boxes is not None:
                for b in r.boxes:
                    if int(b.cls[0]) in {1,2,3,5,7}:
                        boxes.append(tuple(float(v) for v in b.xyxy[0].tolist()))
            overlap = any(iou(a,b)>0.30 for i,a in enumerate(boxes) for b in boxes[i+1:])
            streak = streak + 1 if overlap else 0
            if streak >= 2:
                trigger = idx / fps; break
        idx += 1
    cap.release()
    if trigger is None and idx > 0:
        trigger = (idx / fps) * 0.6  # heuristic mid-clip if YOLO did not fire
    return trigger, idx / fps if fps else 0

trigger_t, total = scan(video_path)
print(f'anomaly trigger at t = {trigger_t:.2f}s of {total:.1f}s total')

In [ ]:
# 5. Extract 4 keyframes around the trigger window
import cv2, numpy as np
from PIL import Image

def keyframes(path, t, pre=5.0, post=2.0, n=4):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    dur = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
    start, end = max(0, t-pre), min(dur, t+post)
    targets = np.linspace(start, end, n)
    out = []
    for ts in targets:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(ts * fps))
        ok, frame = cap.read()
        if ok: out.append((float(ts), Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))))
    cap.release()
    return out

kfs = keyframes(video_path, trigger_t)
print(f'extracted {len(kfs)} keyframes')
kfs[0][1].resize((320, 180))

In [ ]:
# 6. Load LLaVA-1.5-7B in 4-bit (bitsandbytes NF4)
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

model_id = 'llava-hf/llava-1.5-7b-hf'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id, quantization_config=bnb, device_map='auto')
model.eval()
print('loaded', model_id)

In [ ]:
# 7. Forensic analysis on the middle keyframe
import json, re

FORENSIC_PROMPT = '''You are a forensic traffic analyst. The following frame shows a sequence of events captured by a traffic camera.
Analyze and respond in JSON with these exact keys:
{
  "scene_description": "<one paragraph, factual>",
  "vehicles_involved": ["<color> <type>", ...],
  "sequence_of_events": ["<event 1>", "<event 2>", ...],
  "probable_cause": "<one sentence>",
  "violations_observed": ["<violation 1>", ...],
  "at_fault": "<vehicle description or 'unclear'>",
  "confidence": "<low | medium | high>"
}
Be objective. If unclear, say so. Do not invent details. Output only the JSON object.'''

def parse_json(s):
    try: return json.loads(s)
    except Exception: pass
    m = re.search(r'\{[\s\S]*\}', s)
    if m:
        try: return json.loads(m.group(0))
        except Exception: return None
    return None

mid = kfs[len(kfs)//2][1]
conv = [{'role':'user','content':[{'type':'image'},{'type':'text','text':FORENSIC_PROMPT}]}]
prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
inputs = processor(images=mid, text=prompt, return_tensors='pt').to(model.device, torch.float16)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
raw = processor.batch_decode(out, skip_special_tokens=True)[0]
if 'ASSISTANT:' in raw: raw = raw.split('ASSISTANT:')[-1].strip()
print(raw)
forensic = parse_json(raw)
print('parsed:', forensic is not None)

In [ ]:
# 8. Render the incident report
import uuid, datetime
report = forensic or {}
rid = str(uuid.uuid4())[:8]
md = f'''# Incident Report

**Incident ID:** `{rid}`  
**Time of event:** t = {trigger_t:.2f}s  
**Generated:** {datetime.datetime.utcnow().isoformat(timespec="seconds")}Z  
**Model:** llava-hf/llava-1.5-7b-hf (4-bit NF4)

## Scene
{report.get("scene_description","(none)")}

## Vehicles
{chr(10).join("- "+v for v in report.get("vehicles_involved",[])) or "(none)"}

## Sequence
{chr(10).join("- "+v for v in report.get("sequence_of_events",[])) or "(none)"}

## Probable cause
{report.get("probable_cause","unclear")}

## Violations
{chr(10).join("- "+v for v in report.get("violations_observed",[])) or "(none)"}

## Fault
**{report.get("at_fault","unclear")}** (confidence: {report.get("confidence","low")})
'''
from IPython.display import Markdown
Markdown(md)

In [ ]:
# 9. (Optional) Gradio chat UI
import gradio as gr
def chat(q, history):
    q_lower = q.lower()
    if 'fault' in q_lower:
        return f"Fault: {report.get('at_fault','unclear')} (confidence {report.get('confidence','low')})."
    if 'vehicle' in q_lower or 'car' in q_lower:
        return 'Vehicles: ' + ', '.join(report.get('vehicles_involved', [])) or 'No vehicles identified.'
    if 'summary' in q_lower or 'legal' in q_lower:
        return md
    return 'Try asking about fault, vehicles involved, or request a summary.'
gr.ChatInterface(fn=chat).launch(share=False, debug=False)